# 04 — Validation split (Phase 4)

The test set has **no labels**, so we cannot measure our score on it. We need our own "practice exam"
cut out of the training data. This notebook:

1. Explains training data, validation data and **leakage**.
2. Decides *what* to split (S1 entities) and what **not** to split (the S2/S3 search pool).
3. Creates the split and checks it is representative.
4. Tests our scorer (a copy of the official metric) with simple fake predictions.
5. Estimates how many record pairs we would have to compare — the reason blocking exists.
6. Does a dry run of the output format with the **official validator**.

In [1]:
import gc
import inspect
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.evaluation import score_per_entity, macro_f05, score_report
from src.splits import make_validation_split
from src.submission import write_id_list_file

pd.set_option("display.width", 200)
TRAIN_DIR = PROJECT_ROOT / "dataset" / "train"
TEST_DIR = PROJECT_ROOT / "dataset" / "test"

---
## 1. Concepts

### Training data vs validation data

| | Analogy | In our project |
|---|---|---|
| **Training data** | the practice questions you study with | S1 entities whose labels we use to *build* the system (learn features, fit the model) |
| **Validation data** | a mock exam you never studied from | S1 entities whose labels we only use to *measure* the system |
| **Test data** | the real exam | `dataset/test` — no labels, scored by the leaderboard |

If you grade yourself on questions you already studied, you get a flattering score that will not repeat
in the real exam. Same for a model.

### What is leakage?

**Leakage** = information from the validation (or test) labels sneaking into the system we build.
The validation score then looks better than the real test score will be.

Leakage traps specific to our project:

| Trap | Why it leaks | Our rule |
|---|---|---|
| Training the model on pairs of validation S1 entities | the model has seen the answers | model training uses **train** S1 entities only |
| Using the ground truth to build features (e.g. "this S2 record is already matched to someone") | at test time there is no ground truth | features may only use the records' **text** |
| Trying 200 thresholds / feature sets and keeping the best on the same validation set | we slowly "learn" the validation set | iterate on the **small** validation set; confirm on the **full** validation set only occasionally |
| Looking at test data to hand-tune rules | test = real exam | we never look at test labels (there are none) and don't tune on test rows |

Using **unlabeled text** from S2/S3 (e.g. counting how common a word is) is *not* leakage: the same text is
available for the test set too.

---
## 2. What exactly do we split?

The leaderboard scores **each S1 entity as a whole** (all its matches together). So we split
**S1 entities**, never individual pairs: all matches of one entity end up on the same side.

What about Source 2 and Source 3? We do **not** split them. In the test set, each S1 entity searches
all ~10 million S2/S3 records, and 26% of them are "distractors" that match nobody.
If validation S1 entities only searched *their own* matches, the task would be far too easy.
So every S1 entity — train or validation — searches the **full** S2/S3 pool of its country.
Records that belong to *training* entities then act as realistic look-alikes for *validation* entities.

```
                train_source1  (2.2M entities)
               /                              \
      80% "train" entities               20% "validation" entities
      (learn the model)                  (measure the model)
               \                              /
                 both search the FULL S2 + S3 pool
```

### Two validation sizes

- **Full validation**: 20% of S1 = ~441,000 entities. Close to the test size (1.7M). Slow to evaluate.
- **Small validation**: a fixed 20,000-entity subset of it. Fast; used for day-to-day experiments.

---
## 3. Create the split

The function lives in `src/splits.py` because the final pipeline reuses it. Here is its code — plain pandas:

In [2]:
print(inspect.getsource(make_validation_split))

def make_validation_split(ground_truth):
    """Return one row per S1 entity with columns: source1_entity_id, split, small_validation.

    split             "train" or "validation"
    small_validation  True for a fixed 20,000-entity subset of validation, used for fast experiments
    """
    # sort first so the result does not depend on the row order of the file
    all_ids = ground_truth["source1_entity_id"].sort_values().reset_index(drop=True)

    validation_ids = all_ids.sample(frac=VALIDATION_FRACTION, random_state=SEED)
    small_validation_ids = validation_ids.sample(n=SMALL_VALIDATION_SIZE, random_state=SEED)

    split = pd.DataFrame({"source1_entity_id": all_ids})
    split["split"] = "train"
    split.loc[split["source1_entity_id"].isin(validation_ids), "split"] = "validation"
    split["small_validation"] = split["source1_entity_id"].isin(small_validation_ids)
    return split



`random_state=42` makes the random choice **reproducible**: every run picks the same entities.
We sort the IDs first so the result also does not depend on the order of rows in the file.

In [3]:
gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", keep_default_na=False)
split = make_validation_split(gt)
gt = gt.merge(split, on="source1_entity_id")

print(gt["split"].value_counts())
print("small validation entities:", gt["small_validation"].sum())
gt.head()

split
train         1765457
validation     441364
Name: count, dtype: int64
small validation entities: 20000


,source1_entity_id,matched_entity_ids,split,small_validation
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...",train,False
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...",train,False
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467",validation,False
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215",validation,False
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...",train,False


### Is the split representative?

A random split of 2.2M entities should look like the whole dataset. Let's check the things that matter:
country, singletons, number of matches, and how many matches come from S2 vs S3.

In [4]:
s1_country = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", keep_default_na=False, usecols=["entity_id", "country"])
gt = gt.merge(s1_country.rename(columns={"entity_id": "source1_entity_id"}), on="source1_entity_id")
del s1_country
gc.collect()

gt["n_true"] = gt["matched_entity_ids"].map(lambda s: len(s.split(",")) if s else 0)
gt["n_from_s2"] = gt["matched_entity_ids"].str.count("S2-")
gt["n_from_s3"] = gt["matched_entity_ids"].str.count("S3-")

gt["group"] = gt["split"]
gt.loc[gt["small_validation"], "group"] = "small validation"
groups = pd.concat([gt.assign(group="ALL"), gt])

comparison = groups.groupby("group").agg(
    entities=("source1_entity_id", "size"),
    pct_US=("country", lambda c: round((c == "US").mean() * 100, 2)),
    pct_India=("country", lambda c: round((c == "India").mean() * 100, 2)),
    pct_singletons=("n_true", lambda n: round((n == 0).mean() * 100, 2)),
    avg_matches=("n_true", lambda n: round(n.mean(), 3)),
    avg_from_S2=("n_from_s2", lambda n: round(n.mean(), 3)),
    avg_from_S3=("n_from_s3", lambda n: round(n.mean(), 3)),
)
comparison

,entities,pct_US,pct_India,pct_singletons,avg_matches,avg_from_S2,avg_from_S3
group,,,,,,,
ALL,2206821,59.98,40.02,5.58,3.461,1.674,1.788
small validation,20000,59.80,40.20,5.75,3.449,1.667,1.782
train,1765457,59.98,40.02,5.58,3.460,1.673,1.787
validation,421364,59.98,40.02,5.60,3.466,1.676,1.791


In [5]:
(groups.groupby("group")["n_true"].value_counts(normalize=True).unstack(0) * 100).round(2)

group,ALL,small validation,train,validation
n_true,,,,
0,5.58,5.75,5.58,5.60
1,5.40,5.52,5.40,5.38
2,17.00,16.86,17.03,16.90
3,24.05,24.17,24.06,24.04
4,21.94,21.79,21.94,21.95
5,14.59,14.66,14.58,14.62
6,7.47,7.30,7.47,7.48
7,2.90,2.89,2.88,2.96
8,0.85,0.89,0.84,0.86


The groups look alike: same country mix, same share of singletons (~5.6%), same distribution of match counts.
(Note "validation" here means the 420k validation entities *outside* the small subset.)

⚠ **Blind spot:** the training data has no France, so our validation set cannot measure France at all.
We protect ourselves by never writing country-specific rules that only work for US/India.

### Save the split

The split is tiny to recompute, but we also save it so every notebook uses exactly the same one.

In [6]:
splits_dir = PROJECT_ROOT / "output" / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)
split.to_csv(splits_dir / "train_validation_split.tsv", sep="\t", index=False, lineterminator="\n")
print("saved", len(split), "rows to", splits_dir / "train_validation_split.tsv")

saved 2206821 rows to C:\Users\manoj\Desktop\amazon ml challange\output\splits\train_validation_split.tsv


---
## 4. Our scorer — a copy of the official metric

`src/evaluation.py` implements the leaderboard formula (macro F0.5 per S1 entity). Let's read the core:

In [7]:
from src import evaluation
print(inspect.getsource(evaluation.entity_f05))
print(inspect.getsource(evaluation.score_per_entity))

def entity_f05(true_ids, predicted_ids):
    """F0.5 for ONE Source 1 entity (rules from the official problem statement)."""
    if not true_ids and not predicted_ids:
        return 1.0  # correctly predicted "no match"
    correct = len(true_ids & predicted_ids)
    if correct == 0:
        return 0.0
    precision = correct / len(predicted_ids)
    recall = correct / len(true_ids)
    return 1.25 * precision * recall / (0.25 * precision + recall)

def score_per_entity(predictions, truth):
    """One row per Source 1 entity in `truth`, with precision / recall / F0.5.

    Entities missing from `predictions` are treated as "predicted no match".
    """
    table = truth[["source1_entity_id", "matched_entity_ids"]].merge(
        predictions[["source1_entity_id", "matched_entity_ids"]],
        on="source1_entity_id",
        how="left",
        suffixes=("_true", "_pred"),
    )
    true_sets = to_id_sets(table["matched_entity_ids_true"])
    pred_sets = to_id_sets(table["matched_enti

### Sanity checks with fake predictions on the small validation set

A scorer we can't trust is worse than no scorer. We feed it predictions whose score we can reason about:

In [8]:
truth = gt.loc[gt["small_validation"], ["source1_entity_id", "matched_entity_ids"]].reset_index(drop=True)

def keep_first_half(ids):
    ids = ids.split(",") if ids else []
    return ",".join(ids[: (len(ids) + 1) // 2])

def add_one_wrong(ids):
    return ids + ",S2-000000000" if ids else "S2-000000000"

fake_predictions = {
    "perfect answers": truth,
    "predict nothing for everyone": truth.assign(matched_entity_ids=""),
    "only the first half of each true list": truth.assign(matched_entity_ids=truth["matched_entity_ids"].map(keep_first_half)),
    "perfect + one wrong ID for everyone": truth.assign(matched_entity_ids=truth["matched_entity_ids"].map(add_one_wrong)),
}

pd.DataFrame({name: score_report(pred, truth) for name, pred in fake_predictions.items()}).T.round(4)

,macro_F0.5,F0.5 on entities with matches,F0.5 on singletons,pair precision,pair recall,predicted pairs,true pairs
perfect answers,1.0000,1.0000,1.0,1.0000,1.0000,68984.0,68984.0
predict nothing for everyone,0.0575,0.0000,1.0,NaN,0.0000,0.0,68984.0
only the first half of each true list,0.8787,0.8713,1.0,1.0000,0.5687,39232.0,68984.0
perfect + one wrong ID for everyone,0.7499,0.7956,0.0,0.7752,1.0000,88984.0,68984.0


**Reading the table**
- Perfect answers → 1.0 and predicting nothing → the singleton share (~0.056): the scorer behaves as the rules say.
- Finding only **half** of the matches but with no mistakes still scores ~0.88.
- Adding just **one** wrong ID to otherwise perfect answers drops to ~0.75, and every singleton falls from 1 to 0.

That is the whole strategy in one table: **be sure before you predict a match**.

---
## 5. How many pairs would we have to compare?

Comparing every S1 entity with every S2/S3 record **of the same country** (the only safe rule we have so far):

In [9]:
def country_counts(path):
    return pd.read_csv(path, sep="\t", keep_default_na=False, usecols=["country"])["country"].value_counts()

pair_estimate = pd.DataFrame({
    "S1 entities": country_counts(TEST_DIR / "test_source1.tsv"),
    "S2 + S3 records": country_counts(TEST_DIR / "test_source2.tsv") + country_counts(TEST_DIR / "test_source3.tsv"),
})
pair_estimate["pairs to compare"] = pair_estimate["S1 entities"] * pair_estimate["S2 + S3 records"]
pair_estimate.loc["TOTAL"] = pair_estimate.sum()
pair_estimate["pairs to compare"] = pair_estimate["pairs to compare"].map("{:,.0f}".format)
pair_estimate

,S1 entities,S2 + S3 records,pairs to compare
country,,,
India,809986,4717565,"3,821,161,604,090"
US,663106,3817031,"2,531,096,158,286"
France,259452,1434993,"372,311,803,836"
TOTAL,1732544,9969589,"6,724,569,566,212"


About **6.7 trillion** pairs for the test set, even after grouping by country. At a (fast) 1 million
comparisons per second that is ~77 days. **Blocking** (Phase 7) must bring each S1 entity down to a few
dozen candidates — while keeping almost all true matches.

---
## 6. Output format dry run with the official validator

Before building anything, we make sure we can produce files the scorer accepts.
We write the "predict nothing" answer for the **test** set to `output/format_check/` and run
`utils/validate_submission.py` on it.

`src/submission.py` writes with `lineterminator="\n"`: on Windows pandas would otherwise end lines with `\r\n`,
and a strict scorer could read the last ID of a row as `S2-123\r`.

In [10]:
test_s1 = pd.read_csv(TEST_DIR / "test_source1.tsv", sep="\t", keep_default_na=False, usecols=["entity_id"])
empty_answer = pd.DataFrame({
    "source1_entity_id": test_s1["entity_id"],
    "matched_entity_ids": "",
    "candidate_entity_ids": "",
})

check_dir = PROJECT_ROOT / "output" / "format_check"
check_dir.mkdir(parents=True, exist_ok=True)
write_id_list_file(empty_answer, check_dir / "matching_results.tsv", "matched_entity_ids")
write_id_list_file(empty_answer, check_dir / "candidate_pairs.tsv", "candidate_entity_ids")

result = subprocess.run(
    [sys.executable, "utils/validate_submission.py",
     "--matching", "output/format_check/matching_results.tsv",
     "--candidate", "output/format_check/candidate_pairs.tsv",
     "--test-dir", "dataset/test"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},   # Windows consoles default to cp1252
)
print(result.stdout)
print("exit code:", result.returncode)

ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (1732544 empty, 0 non-empty).
  candidate_pairs.tsv: 1732544 rows (1732544 empty, 0 non-empty).

PASS — no blocking issues found. Safe to submit.

exit code: 0


**PASS** — the format is right. (This empty answer would score only ~0.056, so we won't submit it.)

---
# Summary — Phase 4

| Decision | Why |
|---|---|
| Split **S1 entities** 80 / 20 (seed 42) | the score is per S1 entity; all its matches stay together |
| S2/S3 **not** split — everyone searches the full pool | keeps distractors realistic, like the test set |
| Small validation = 20,000 entities | fast experiments; full validation (441k) only to confirm |
| Model training uses train entities only; features use text only | avoids leakage |
| Scorer `src/evaluation.py` = official macro F0.5 | checked with fake predictions |
| Writer `src/submission.py` with `\n` line endings | official validator: PASS |

**Floor to beat on validation: 0.056 (predict nothing).**